# Trust Region Methods: From REINFORCE to TRPO to PPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/trpo_ppo.ipynb)

This notebook implements three policy gradient methods from scratch in PyTorch and compares them on CartPole-v1:

1. **REINFORCE** with baseline — the simplest policy gradient
2. **TRPO** — Trust Region Policy Optimization with conjugate gradient + line search
3. **PPO-Clip** — Proximal Policy Optimization with clipped surrogate objective

All hyperparameters are taken from John Schulman's original `modular_rl` codebase.

**Blog post**: [Trust Region Methods: From REINFORCE to TRPO to PPO](https://sesen.ai/blog/trust-region-methods-reinforce-trpo-ppo)

In [ ]:
!pip install gymnasium torch matplotlib -q

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Shared Components

All three methods share the same network architecture, rollout collection, and advantage computation.

### Network Architecture

From the original `agentzoo.py`: two hidden layers of 64 units with tanh activation.

In [ ]:
class PolicyNetwork(nn.Module):
    """2-layer MLP [64,64] tanh -> softmax (from agentzoo.py)."""
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, act_dim),
            nn.Softmax(dim=-1),
        )
        # Small init on the output Linear layer (before Softmax)
        self.net[-2].weight.data *= 0.1
        self.net[-2].bias.data *= 0.1

    def forward(self, x):
        return self.net(x)


class ValueNetwork(nn.Module):
    """2-layer MLP [64,64] tanh -> linear (from agentzoo.py)."""
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

### Rollout Collection and GAE

Collect experience in batches of `timesteps_per_batch=10000` timesteps (from `core.py`), then compute Generalised Advantage Estimation with `gamma=0.99` and `lambda=1.0`.

In [ ]:
def collect_rollouts(env, policy, timesteps_per_batch=10000):
    """Collect rollouts until we have >= timesteps_per_batch timesteps."""
    observations, actions, rewards, dones, log_probs = [], [], [], [], []
    episode_rewards = []
    timesteps = 0

    while timesteps < timesteps_per_batch:
        obs, _ = env.reset()
        episode_reward = 0
        done = False

        while not done:
            obs_t = torch.FloatTensor(obs)
            with torch.no_grad():
                probs = policy(obs_t)
            dist = Categorical(probs)
            action = dist.sample()

            observations.append(obs)
            actions.append(action.item())
            log_probs.append(dist.log_prob(action).item())

            obs, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            rewards.append(reward)
            dones.append(done)
            episode_reward += reward
            timesteps += 1

        episode_rewards.append(episode_reward)

    return {
        'observations': np.array(observations),
        'actions': np.array(actions),
        'rewards': np.array(rewards),
        'dones': np.array(dones),
        'log_probs': np.array(log_probs),
        'episode_rewards': episode_rewards,
    }


def compute_gae(rewards, dones, values, gamma=0.99, lam=1.0):
    """Generalised Advantage Estimation (core.py line 49-73)."""
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_advantage = 0
    last_value = 0

    for t in reversed(range(len(rewards))):
        if dones[t]:
            last_advantage = 0
            last_value = 0
        delta = rewards[t] + gamma * last_value - values[t]
        advantages[t] = last_advantage = delta + gamma * lam * last_advantage
        last_value = values[t]

    returns = advantages + values
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns

## Method 1: REINFORCE with Baseline

The simplest policy gradient method. Compute the gradient and take a step — no constraints, no trust region.

In [ ]:
def train_reinforce(n_iters=200, timesteps_per_batch=10000, gamma=0.99,
                    lam=1.0, lr=1e-3, seed=SEED):
    env = gym.make('CartPole-v1')
    env.reset(seed=seed)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    policy = PolicyNetwork(obs_dim, act_dim)
    value_fn = ValueNetwork(obs_dim)
    policy_opt = optim.Adam(policy.parameters(), lr=lr)
    value_opt = optim.Adam(value_fn.parameters(), lr=1e-3)

    mean_rewards = []
    kl_divs = []

    for iteration in range(n_iters):
        rollout = collect_rollouts(env, policy, timesteps_per_batch)
        obs_t = torch.FloatTensor(rollout['observations'])
        acts_t = torch.LongTensor(rollout['actions'])

        with torch.no_grad():
            values = value_fn(obs_t).numpy()
            old_probs = policy(obs_t)

        advantages, returns = compute_gae(
            rollout['rewards'], rollout['dones'], values, gamma, lam
        )
        adv_t = torch.FloatTensor(advantages)
        ret_t = torch.FloatTensor(returns)

        # Policy update
        probs = policy(obs_t)
        dist = Categorical(probs)
        log_probs = dist.log_prob(acts_t)
        policy_loss = -(log_probs * adv_t).mean()

        policy_opt.zero_grad()
        policy_loss.backward()
        policy_opt.step()

        # Value function update
        for _ in range(10):
            v_pred = value_fn(obs_t)
            v_loss = ((v_pred - ret_t) ** 2).mean()
            value_opt.zero_grad()
            v_loss.backward()
            value_opt.step()

        # KL divergence tracking
        with torch.no_grad():
            new_probs = policy(obs_t)
            kl = (old_probs * (torch.log(old_probs + 1e-8)
                  - torch.log(new_probs + 1e-8))).sum(dim=-1).mean().item()

        mean_reward = np.mean(rollout['episode_rewards'])
        mean_rewards.append(mean_reward)
        kl_divs.append(kl)

        if (iteration + 1) % 20 == 0:
            print(f"REINFORCE iter {iteration+1}: "
                  f"mean_reward={mean_reward:.1f}, KL={kl:.4f}")

    env.close()
    return mean_rewards, kl_divs

## Method 2: TRPO

Same objective but constrained by KL divergence. Uses conjugate gradient to compute the natural gradient direction, then a line search to find the step size.

In [ ]:
def conjugate_gradient(f_Ax, b, cg_iters=10, residual_tol=1e-10):
    """Solve Hx = b using conjugate gradient (trpo.py line 129-161)."""
    p = b.clone()
    r = b.clone()
    x = torch.zeros_like(b)
    rdotr = r.dot(r)

    for _ in range(cg_iters):
        z = f_Ax(p)
        v = rdotr / (p.dot(z) + 1e-8)
        x += v * p
        r -= v * z
        newrdotr = r.dot(r)
        mu = newrdotr / (rdotr + 1e-8)
        p = r + mu * p
        rdotr = newrdotr
        if rdotr < residual_tol:
            break
    return x


def linesearch(f, x, fullstep, expected_improve_rate,
               max_backtracks=10, accept_ratio=0.1):
    """Backtracking line search (trpo.py line 110-127)."""
    fval = f(x)
    for stepfrac in 0.5 ** np.arange(max_backtracks):
        xnew = x + stepfrac * fullstep
        newfval = f(xnew)
        actual_improve = fval - newfval
        expected_improve = expected_improve_rate * stepfrac
        ratio = actual_improve / (expected_improve + 1e-8)
        if ratio > accept_ratio and actual_improve > 0:
            return True, xnew
    return False, x


def flat_params(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])


def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        size = p.numel()
        p.data.copy_(flat[offset:offset+size].view(p.shape))
        offset += size

In [ ]:
def train_trpo(n_iters=200, timesteps_per_batch=10000, gamma=0.99, lam=1.0,
               max_kl=0.01, cg_damping=1e-3, cg_iters=10, seed=SEED):
    env = gym.make('CartPole-v1')
    env.reset(seed=seed)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    policy = PolicyNetwork(obs_dim, act_dim)
    value_fn = ValueNetwork(obs_dim)
    value_opt = optim.Adam(value_fn.parameters(), lr=1e-3)

    mean_rewards = []
    kl_divs = []

    for iteration in range(n_iters):
        rollout = collect_rollouts(env, policy, timesteps_per_batch)
        obs_t = torch.FloatTensor(rollout['observations'])
        acts_t = torch.LongTensor(rollout['actions'])

        with torch.no_grad():
            values = value_fn(obs_t).numpy()

        advantages, returns = compute_gae(
            rollout['rewards'], rollout['dones'], values, gamma, lam
        )
        adv_t = torch.FloatTensor(advantages)
        ret_t = torch.FloatTensor(returns)

        old_params = flat_params(policy).clone()
        with torch.no_grad():
            old_probs = policy(obs_t).detach()

        # Surrogate loss and gradient
        probs = policy(obs_t)
        dist = Categorical(probs)
        log_probs = dist.log_prob(acts_t)
        old_dist = Categorical(old_probs)
        old_log_probs = old_dist.log_prob(acts_t)

        ratio = torch.exp(log_probs - old_log_probs)
        surr_loss = -(ratio * adv_t).mean()

        grads = torch.autograd.grad(surr_loss, policy.parameters())
        pg = torch.cat([g.view(-1) for g in grads]).detach()

        if torch.allclose(pg, torch.zeros_like(pg)):
            mean_rewards.append(np.mean(rollout['episode_rewards']))
            kl_divs.append(0.0)
            continue

        # Fisher-vector product
        def fvp(v):
            probs_ = policy(obs_t)
            kl = (old_probs * (torch.log(old_probs + 1e-8)
                  - torch.log(probs_ + 1e-8))).sum(dim=-1).mean()
            grads_ = torch.autograd.grad(kl, policy.parameters(),
                                         create_graph=True)
            flat_grad = torch.cat([g.view(-1) for g in grads_])
            kl_v = (flat_grad * v).sum()
            grads2 = torch.autograd.grad(kl_v, policy.parameters())
            return torch.cat([g.view(-1) for g in grads2]).detach() \
                   + cg_damping * v

        # CG to find natural gradient direction
        stepdir = conjugate_gradient(fvp, -pg, cg_iters=cg_iters)

        # Step size from trust region constraint
        shs = 0.5 * stepdir.dot(fvp(stepdir))
        lm = torch.sqrt(shs / max_kl)
        fullstep = stepdir / lm

        # Line search
        neggdotstepdir = -pg.dot(stepdir)
        def surrogate_loss(params):
            set_flat_params(policy, params)
            with torch.no_grad():
                p = policy(obs_t)
                d = Categorical(p)
                lp = d.log_prob(acts_t)
                r = torch.exp(lp - old_log_probs.detach())
                return -(r * adv_t).mean().item()

        success, new_params = linesearch(
            surrogate_loss, old_params, fullstep, neggdotstepdir / lm
        )
        set_flat_params(policy, new_params)

        # Value function update
        for _ in range(10):
            v_pred = value_fn(obs_t)
            v_loss = ((v_pred - ret_t) ** 2).mean()
            value_opt.zero_grad()
            v_loss.backward()
            value_opt.step()

        # KL divergence tracking
        with torch.no_grad():
            new_probs = policy(obs_t)
            kl = (old_probs * (torch.log(old_probs + 1e-8)
                  - torch.log(new_probs + 1e-8))).sum(dim=-1).mean().item()

        mean_reward = np.mean(rollout['episode_rewards'])
        mean_rewards.append(mean_reward)
        kl_divs.append(kl)

        if (iteration + 1) % 20 == 0:
            print(f"TRPO iter {iteration+1}: "
                  f"mean_reward={mean_reward:.1f}, KL={kl:.4f}")

    env.close()
    return mean_rewards, kl_divs

## Method 3: PPO-Clip

Replace the constrained optimization with a clipped surrogate objective. Multiple epochs of minibatch updates on the same data.

In [ ]:
def train_ppo(n_iters=200, timesteps_per_batch=10000, gamma=0.99, lam=1.0,
              clip_epsilon=0.2, epochs=10, lr=1e-3, seed=SEED):
    env = gym.make('CartPole-v1')
    env.reset(seed=seed)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    policy = PolicyNetwork(obs_dim, act_dim)
    value_fn = ValueNetwork(obs_dim)
    policy_opt = optim.Adam(policy.parameters(), lr=lr)
    value_opt = optim.Adam(value_fn.parameters(), lr=1e-3)

    mean_rewards = []
    kl_divs = []

    for iteration in range(n_iters):
        rollout = collect_rollouts(env, policy, timesteps_per_batch)
        obs_t = torch.FloatTensor(rollout['observations'])
        acts_t = torch.LongTensor(rollout['actions'])

        with torch.no_grad():
            values = value_fn(obs_t).numpy()
            old_probs = policy(obs_t).detach()
            old_dist = Categorical(old_probs)
            old_log_probs = old_dist.log_prob(acts_t).detach()

        advantages, returns = compute_gae(
            rollout['rewards'], rollout['dones'], values, gamma, lam
        )
        adv_t = torch.FloatTensor(advantages)
        ret_t = torch.FloatTensor(returns)

        # Multiple epochs of minibatch updates
        batch_size = 128
        n_samples = len(obs_t)

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            for start in range(0, n_samples, batch_size):
                end = start + batch_size
                idx = indices[start:end]

                mb_obs = obs_t[idx]
                mb_acts = acts_t[idx]
                mb_adv = adv_t[idx]
                mb_old_lp = old_log_probs[idx]
                mb_ret = ret_t[idx]

                # Clipped surrogate objective
                probs = policy(mb_obs)
                dist = Categorical(probs)
                log_probs = dist.log_prob(mb_acts)

                ratio = torch.exp(log_probs - mb_old_lp)
                surr1 = ratio * mb_adv
                surr2 = torch.clamp(ratio, 1 - clip_epsilon,
                                    1 + clip_epsilon) * mb_adv
                policy_loss = -torch.min(surr1, surr2).mean()

                policy_opt.zero_grad()
                policy_loss.backward()
                policy_opt.step()

                # Value loss
                v_pred = value_fn(mb_obs)
                v_loss = ((v_pred - mb_ret) ** 2).mean()
                value_opt.zero_grad()
                v_loss.backward()
                value_opt.step()

        # KL divergence tracking
        with torch.no_grad():
            new_probs = policy(obs_t)
            kl = (old_probs * (torch.log(old_probs + 1e-8)
                  - torch.log(new_probs + 1e-8))).sum(dim=-1).mean().item()

        mean_reward = np.mean(rollout['episode_rewards'])
        mean_rewards.append(mean_reward)
        kl_divs.append(kl)

        if (iteration + 1) % 20 == 0:
            print(f"PPO iter {iteration+1}: "
                  f"mean_reward={mean_reward:.1f}, KL={kl:.4f}")

    env.close()
    return mean_rewards, kl_divs

## Train All Three

Each method trains for 200 iterations with batches of 10,000 timesteps.

In [ ]:
print("Training REINFORCE...")
reinforce_rewards, reinforce_kl = train_reinforce(n_iters=200)

print("\nTraining TRPO...")
trpo_rewards, trpo_kl = train_trpo(n_iters=200)

print("\nTraining PPO...")
ppo_rewards, ppo_kl = train_ppo(n_iters=200)

## Compare Reward Curves

In [ ]:
def smooth(data, window=10):
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

window = 20
for rewards, color, label in [
    (reinforce_rewards, '#e74c3c', 'REINFORCE'),
    (trpo_rewards, '#3498db', 'TRPO'),
    (ppo_rewards, '#2ecc71', 'PPO'),
]:
    ax.plot(rewards, color=color, alpha=0.2, linewidth=0.8)
    s = smooth(rewards, window)
    ax.plot(range(window-1, len(rewards)), s, color=color,
            linewidth=2.5, label=label)

ax.axhline(y=475, color='gray', linestyle='--', alpha=0.5, label='Solved (475)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Mean Episode Reward', fontsize=12)
ax.set_title('REINFORCE vs TRPO vs PPO on CartPole-v1', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(0, 520)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Compare KL Divergence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, kl_data, color) in zip(axes, [
    ('REINFORCE', reinforce_kl, '#e74c3c'),
    ('TRPO', trpo_kl, '#3498db'),
    ('PPO', ppo_kl, '#2ecc71'),
]):
    kl_arr = np.array(kl_data)
    ax.plot(kl_arr, color=color, alpha=0.4, linewidth=0.8)
    if len(kl_arr) > 10:
        kl_smooth = smooth(kl_arr, 10)
        ax.plot(range(9, len(kl_arr)), kl_smooth, color=color, linewidth=2)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('KL Divergence', fontsize=11)
    if name != 'REINFORCE':
        ax.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5,
                   label='Target (0.01)')
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('KL Divergence Per Update', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Visualise the PPO Clipped Objective

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
epsilon = 0.2
ratios = np.linspace(0.5, 1.8, 500)

for ax, (A, label) in zip(axes, [
    (1.0, 'Positive Advantage (A > 0)'),
    (-1.0, 'Negative Advantage (A < 0)'),
]):
    surr1 = ratios * A
    surr2 = np.clip(ratios, 1-epsilon, 1+epsilon) * A
    clipped = np.minimum(surr1, surr2)

    ax.plot(ratios, surr1, 'b--', alpha=0.5, linewidth=1.5,
            label='Unclipped: r(θ)A')
    ax.plot(ratios, clipped, color='#2ecc71', linewidth=2.5,
            label='Clipped objective')
    ax.axvline(x=1-epsilon, color='gray', linestyle=':', alpha=0.7)
    ax.axvline(x=1+epsilon, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel('Policy Ratio r(θ)', fontsize=10)
    ax.set_ylabel('Objective', fontsize=10)
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('PPO Clipped Surrogate Objective (ε = 0.2)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Exercises

1. **Try LunarLander-v3**: Replace `CartPole-v1` with `LunarLander-v3` (install with `pip install gymnasium[box2d]`). It has 8-dimensional observations and 4 discrete actions. PPO should still converge reliably.

2. **Tune the clip epsilon**: Try `clip_epsilon=0.1` and `clip_epsilon=0.3`. Smaller values make PPO more conservative; larger values allow bigger updates.

3. **Implement the adaptive KL penalty variant**: Instead of clipping, add a KL penalty to the loss:
   ```python
   kl_coeff = 1.0  # adaptive coefficient
   kl_target = 0.01
   loss = surrogate_loss + kl_coeff * kl_divergence
   # After each iteration:
   if kl > 1.3 * kl_target: kl_coeff *= 1.5
   elif kl < 0.7 * kl_target: kl_coeff /= 1.5
   ```

4. **Compare wall-clock time**: Add timing around each training loop and compare seconds per iteration for REINFORCE, TRPO, and PPO.

5. **Vary GAE lambda**: Try `lam=0.95` instead of `lam=1.0`. This adds bias but reduces variance — does it help PPO converge faster?